# Lezione 39 — La matematica di LoRA

> **Modulo:** LoRA e adattamento efficiente · **Tempo stimato:** 30 minuti
> **Prerequisiti:** Lezione 8 (gradienti), Lezione 38 (freezing).
>
> Obiettivo pratico unico: capire e implementare la **decomposizione a basso
> rango** $W = W_0 + BA$ e misurare il risparmio di parametri.

## Teoria essenziale

LoRA (Hu et al., 2021) parte da un'osservazione: quando adatti un grande strato
$W_0$ (dimensione $d\times k$) a un nuovo compito, l'**aggiornamento** $\Delta W$
necessario ha spesso *rango basso*. Invece di imparare tutto $\Delta W$
($d\cdot k$ parametri), lo si scrive come prodotto di due matrici piccole:

$$W = W_0 + \Delta W = W_0 + B A,\quad B\in\mathbb{R}^{d\times r},\ A\in\mathbb{R}^{r\times k},\ r\ll\min(d,k)$$

$W_0$ resta **congelato**; si addestrano solo $A$ e $B$. I parametri passano da
$d\cdot k$ a $r\,(d+k)$: per $r$ piccolo, un crollo.

In [1]:
import numpy as np

rng = np.random.default_rng(39)

d, k = 64, 48
for r in [1, 2, 4, 8, 16]:
    pieni = d * k
    lora = r * (d + k)
    print(f"r={r:2d}: LoRA {lora:5d} param vs pieni {pieni} "
          f"-> {100*lora/pieni:5.1f}% ({pieni/lora:.1f}x meno)")

r= 1: LoRA   112 param vs pieni 3072 ->   3.6% (27.4x meno)
r= 2: LoRA   224 param vs pieni 3072 ->   7.3% (13.7x meno)
r= 4: LoRA   448 param vs pieni 3072 ->  14.6% (6.9x meno)
r= 8: LoRA   896 param vs pieni 3072 ->  29.2% (3.4x meno)
r=16: LoRA  1792 param vs pieni 3072 ->  58.3% (1.7x meno)


## Ricostruire un aggiornamento a basso rango

Se $\Delta W$ e' *davvero* di rango basso, $BA$ puo' ricostruirlo quasi
perfettamente. Costruiamo un $\Delta W$ di rango 3 e vediamo che una LoRA con
$r=3$ lo cattura, mentre $r=1$ no.

In [2]:
d, k, rango_vero = 32, 24, 3

# Delta W di rango esattamente 3 = prodotto di due fattori 3-dimensionali
B_vero = rng.normal(size=(d, rango_vero))
A_vero = rng.normal(size=(rango_vero, k))
Delta = B_vero @ A_vero

def miglior_approx_rango(M, r):
    # migliore approssimazione di rango r via SVD (teorema di Eckart-Young)
    U, S, Vt = np.linalg.svd(M, full_matrices=False)
    return (U[:, :r] * S[:r]) @ Vt[:r]

for r in [1, 2, 3, 5]:
    approx = miglior_approx_rango(Delta, r)
    errore = np.linalg.norm(Delta - approx) / np.linalg.norm(Delta)
    print(f"r={r}: errore relativo di ricostruzione = {errore:.4f}")

r=1: errore relativo di ricostruzione = 0.6153
r=2: errore relativo di ricostruzione = 0.3509
r=3: errore relativo di ricostruzione = 0.0000
r=5: errore relativo di ricostruzione = 0.0000


Leggi gli errori: con $r$ pari al rango vero (3) l'errore crolla a ~0 — la
LoRA cattura *esattamente* l'aggiornamento. Con $r<3$ resta un errore: non c'e'
abbastanza capacita'. E' il compromesso centrale di LoRA: $r$ controlla quanta
"stoffa" ha l'adattamento.

## Il progetto: Memory AI Lab, passo 39

Impacchetto una funzione che, dati $d,k,r$, restituisce il risparmio di
parametri — la useremo per giustificare la scelta di $r$ quando adatteremo il
classificatore di memorie.

In [3]:
def risparmio_lora(d, k, r):
    pieni = d * k
    lora = r * (d + k)
    return {"pieni": pieni, "lora": lora,
            "percentuale": round(100 * lora / pieni, 2),
            "fattore": round(pieni / lora, 2)}

r_info = risparmio_lora(64, 48, 4)
assert r_info["lora"] == 4 * (64 + 48)
assert r_info["lora"] < r_info["pieni"]
print("esempio (d=64,k=48,r=4):", r_info)

esempio (d=64,k=48,r=4): {'pieni': 3072, 'lora': 448, 'percentuale': 14.58, 'fattore': 6.86}


## Riepilogo (max 8 punti)

1. L'aggiornamento $\Delta W$ per un nuovo compito e' spesso di **rango basso**.
2. LoRA lo scrive come $BA$ con $B\,(d\times r)$, $A\,(r\times k)$, $r\ll d,k$.
3. $W_0$ resta congelato; si addestrano solo $A$ e $B$.
4. Parametri: da $d\cdot k$ a $r(d+k)$ → crollo per $r$ piccolo.
5. Se $\Delta W$ ha rango $\le r$, $BA$ lo ricostruisce (quasi) esattamente.
6. Se $r$ e' troppo piccolo, resta un errore di ricostruzione.
7. La miglior approssimazione di rango $r$ e' data dalla **SVD** (Eckart-Young).
8. $r$ e' il pomello capacita'/costo dell'adattamento.

## Quiz

1. Quanti parametri ha una LoRA con $d=100,k=100,r=4$? E il pieno?
2. Cosa succede all'errore di ricostruzione se $r \ge$ rango di $\Delta W$?
3. Perche' $W_0$ non va addestrato?

*(Risposte: 1. LoRA $4\cdot200=800$ vs pieno $10000$; 2. tende a zero; 3. e'
il modello pre-addestrato che vogliamo preservare — congelarlo evita il
forgetting e azzera il suo costo di gradiente.)*